# Attr-Only Elbow Like-Game Clustering

This notebook clusters games using `game_attr` attributes only. It intentionally excludes cabinet-name features and performance-derived features from the clustering matrix.

The relationship map is used only to keep accepted attr-to-historical-game links and to produce the final like-game mapping table. Low-confidence review/fuzzy/no-match relationship rows are dropped before matching.

Cluster count is selected with an elbow view. The default chosen cluster count is 8 because that looked more useful for review than maximizing silhouette.


In [0]:
from pathlib import Path
import ast
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.spatial.distance import pdist, squareform

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 220)

ROOT = Path.cwd()
if not (ROOT / "Data").exists() and (ROOT.parent / "Data").exists():
    ROOT = ROOT.parent

ATTR_PATH = ROOT / "Data" / "processed" / "game_attr_cleaned.csv"
ATTR_RAW_PATH = ROOT / "Data" / "raw sample" / "game_attr (2).csv"
REL_MAP_PATH = ROOT / "Data" / "processed" / "attr_clean_to_perf_best_match_mapping_all_best_candidates.csv"

OUTPUT_DIR = ROOT / "Data" / "processed" / "reports" / "attr_only_elbow_clustering"
CLUSTER_CHART_DIR = OUTPUT_DIR / "cluster_feature_charts"
HOLDOUT_CHART_DIR = OUTPUT_DIR / "holdout_feature_charts"
for path in [OUTPUT_DIR, CLUSTER_CHART_DIR, HOLDOUT_CHART_DIR]:
    path.mkdir(parents=True, exist_ok=True)

PROFILES_OUTPUT_PATH = ROOT / "Data" / "processed" / "attr_only_elbow_profiles.csv"
FEATURE_MATRIX_OUTPUT_PATH = ROOT / "Data" / "processed" / "attr_only_elbow_features.csv"
MAPPING_OUTPUT_PATH = ROOT / "Data" / "processed" / "attr_only_elbow_like_game_mapping.csv"
GROUPED_MAPPING_OUTPUT_PATH = ROOT / "Data" / "processed" / "attr_only_elbow_like_game_mapping_grouped.csv"
CLUSTER_SUMMARY_OUTPUT_PATH = ROOT / "Data" / "processed" / "attr_only_elbow_cluster_summary.csv"
ELBOW_OUTPUT_PATH = OUTPUT_DIR / "elbow_scores.csv"
CHART_INDEX_OUTPUT_PATH = OUTPUT_DIR / "chart_index.csv"
MANIFEST_OUTPUT_PATH = OUTPUT_DIR / "manifest.json"

print(f"Project root: {ROOT}")
print(f"Attr input: {ATTR_PATH}")
print(f"Relationship map: {REL_MAP_PATH}")

In [0]:
HOLDOUT_FRACTION = 0.10
MIN_HOLDOUT_ROWS = 1
MAX_LIKE_GAMES_PER_TARGET = None  # None keeps every historical attr game inside the smallest qualifying dendrogram cluster.

LINKAGE_METHOD = "complete"
DISTANCE_METRIC = "cosine"
ELBOW_MIN_CLUSTERS = 2
ELBOW_MAX_CLUSTERS = 16
ELBOW_SELECTED_N_CLUSTERS = 8
MAX_DENDROGRAM_LEAVES = 90
MAX_TEXT_TOKENS = 140
TOP_CLUSTER_FEATURES = 14
TOP_HOLDOUT_FEATURES = 16

EXCLUDED_MATCH_METHODS = {
    "possible_review_fuzzy",
    "weak_review_only",
    "no_match",
}

ATTR_FEATURE_MULTIPLIER = 2.0
TEXT_TOKEN_WEIGHT = 0.70
PROFILE_BIAS_WEIGHT = 0.01

In [0]:
attr = pd.read_csv(ATTR_PATH, low_memory=False)
rel = pd.read_csv(REL_MAP_PATH, low_memory=False)

raw_extra_columns = [
    "theme_sk",
    "family_nk",
    "brand_nk",
    "brand_name",
    "cert_family_ref",
    "business_segment",
    "lines",
    "ways",
    "gaming_channel_cd",
    "volatility_cd",
    "progressive_tiers",
    "na_release_date",
    "lac_release_date",
    "emea_release_date",
    "apac_release_date",
]
if ATTR_RAW_PATH.exists():
    attr_raw = pd.read_csv(ATTR_RAW_PATH, usecols=lambda column: column in raw_extra_columns, low_memory=False)
    attr_raw = attr_raw.drop_duplicates(subset=["theme_sk"])
    extra_cols = [column for column in attr_raw.columns if column != "theme_sk"]
    attr = attr.merge(attr_raw[["theme_sk", *extra_cols]], on="theme_sk", how="left")

print(f"Attr rows: {len(attr):,}; columns after raw extras: {len(attr.columns):,}")
print(f"Relationship-map rows before filter: {len(rel):,}")
display(attr.head(3))
display(rel.head(3))

In [0]:
def norm_text(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text if text else pd.NA


def clean_category(value):
    text = norm_text(value)
    if pd.isna(text):
        return "__missing__"
    text = re.sub(r"[\[\]'\\\"]", "", str(text))
    text = re.sub(r"\s+", " ", text).strip()
    return text if text else "__missing__"


def numeric_series(series):
    cleaned = (
        series.astype("string")
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip()
    )
    return pd.to_numeric(cleaned, errors="coerce")


def bool_series(series):
    truthy = {"true", "1", "y", "yes", "t"}
    falsy = {"false", "0", "n", "no", "f"}

    def convert(value):
        if pd.isna(value):
            return np.nan
        if isinstance(value, bool):
            return 1.0 if value else 0.0
        text = str(value).strip().lower()
        if text in truthy:
            return 1.0
        if text in falsy:
            return 0.0
        number = pd.to_numeric(text, errors="coerce")
        if pd.isna(number):
            return np.nan
        return 1.0 if float(number) != 0 else 0.0

    return series.map(convert).astype(float)


def list_unique(series, limit=8):
    values = []
    seen = set()
    for value in series.dropna().astype(str):
        cleaned = value.strip()
        key = cleaned.lower()
        if not cleaned or key in seen:
            continue
        values.append(cleaned)
        seen.add(key)
        if len(values) >= limit:
            break
    return " | ".join(values)


def parse_listish(value):
    if pd.isna(value):
        return []
    if isinstance(value, (list, tuple, set)):
        return [str(item).strip() for item in value if str(item).strip()]
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (list, tuple, set)):
            return [str(item).strip() for item in parsed if str(item).strip()]
    except Exception:
        pass
    return [piece.strip() for piece in re.split(r"[,;|/]+", text) if piece.strip()]


def months_between(later, earlier):
    if pd.isna(later) or pd.isna(earlier):
        return np.nan
    return (later.year - earlier.year) * 12 + (later.month - earlier.month)


def release_age_bucket(months):
    if pd.isna(months):
        return "unknown_age"
    if months < 6:
        return "0_5_months"
    if months < 12:
        return "6_11_months"
    if months < 24:
        return "12_23_months"
    if months < 48:
        return "24_47_months"
    if months < 84:
        return "48_83_months"
    return "84_plus_months"


def robust_scale_01(series):
    values = numeric_series(series).astype(float)
    if values.notna().sum() == 0:
        return None
    filled = values.fillna(values.median())
    low = filled.quantile(0.01)
    high = filled.quantile(0.99)
    if not np.isfinite(low) or not np.isfinite(high) or high <= low:
        return None
    return ((filled.clip(low, high) - low) / (high - low)).fillna(0.0)


STOPWORDS = {
    "the", "and", "for", "with", "from", "this", "that", "game", "games", "slot", "slots",
    "deluxe", "edition", "everi", "igt", "inc", "llc", "new", "machine", "used",
    "exclude", "unknown", "missing", "theme", "standard", "video",
}


def tokenize_text(value):
    text = clean_category(value).replace("_", " ")
    tokens = re.findall(r"[a-z0-9]+", text.lower())
    return [token for token in tokens if len(token) >= 3 and token not in STOPWORDS and not token.isdigit()]


def pretty_feature_name(name):
    text = str(name)
    for prefix in ["cat__", "bool__", "num__", "token__"]:
        text = text.replace(prefix, "")
    text = text.replace("__missing__", "missing")
    text = text.replace("_", " ")
    return text[:68]


def safe_filename(value, max_len=80):
    text = re.sub(r"[^a-zA-Z0-9]+", "_", str(value)).strip("_").lower()
    if not text:
        text = "item"
    return text[:max_len]

In [0]:
rel_work = rel.copy()
rel_work["join_attr_theme_name"] = rel_work["theme_name_friendly"].map(norm_text)
rel_work["match_method_norm"] = rel_work["match_method"].map(norm_text)
rel_work["matched_historical_game_name"] = rel_work["perf_game_name"].map(norm_text)
rel_work["match_score_num"] = numeric_series(rel_work["match_score"])
rel_work["matched_release_date"] = pd.to_datetime(rel_work["game_release_date"], errors="coerce")

rel_filtered = rel_work[
    rel_work["join_attr_theme_name"].notna()
    & rel_work["matched_historical_game_name"].notna()
    & ~rel_work["match_method_norm"].isin(EXCLUDED_MATCH_METHODS)
].copy()

method_priority = {
    "raw_exact": 1,
    "basic_clean_exact": 2,
    "aggressive_clean_exact": 3,
    "very_likely_fuzzy": 4,
}
rel_filtered["method_priority"] = rel_filtered["match_method_norm"].map(method_priority).fillna(99)
rel_filtered = rel_filtered.sort_values(
    ["join_attr_theme_name", "method_priority", "match_score_num"],
    ascending=[True, True, False],
)

rel_profile = (
    rel_filtered.groupby("join_attr_theme_name", as_index=False)
    .agg(
        best_historical_game_name=("matched_historical_game_name", "first"),
        historical_game_names=("matched_historical_game_name", lambda s: list_unique(s, limit=8)),
        historical_candidate_count=("matched_historical_game_name", "nunique"),
        accepted_match_methods=("match_method_norm", lambda s: list_unique(s, limit=6)),
        best_match_method=("match_method_norm", "first"),
        best_match_score=("match_score_num", "first"),
        matched_release_date=("matched_release_date", "first"),
    )
)

print(f"Relationship-map rows after filter: {len(rel_filtered):,}")
print(f"Accepted attr themes in relationship map: {rel_profile['join_attr_theme_name'].nunique():,}")
print(rel_filtered["match_method_norm"].value_counts().to_string())
display(rel_profile.head(5))

In [0]:
def derive_attr_theme_bucket(row):
    text = " ".join(
        str(row.get(column, ""))
        for column in ["theme_name", "theme_name_friendly", "family_name", "brand_name", "product_family", "game_type"]
        if pd.notna(row.get(column, pd.NA))
    ).lower()
    buckets = {
        "ancient_civilizations": ["egypt", "egyptian", "incan", "mayan", "aztec", "roman", "greek", "ramosis", "cleopatra"],
        "fantasy_mythology": ["fantasy", "myth", "genie", "fairy", "fairies", "dragon", "magic", "wizard", "phoenix"],
        "nature_animals": ["animal", "buffalo", "wolf", "tiger", "lion", "cat", "dog", "fish", "ocean", "sea", "jungle", "forest", "island", "eagle", "bear"],
        "adventure_exploration": ["pirate", "viking", "treasure", "quest", "voyage", "adventure", "expedition", "frontier"],
        "entertainment_characters": ["movie", "tv", "music", "show", "star", "celebrity", "cartoon", "character", "carnival", "carnivale"],
        "culture_regional": ["asian", "china", "chinese", "japan", "irish", "italian", "latin", "western", "country", "native", "australian"],
        "luxury_classic": ["gold", "diamond", "gem", "wealth", "cash", "money", "royal", "king", "queen", "classic", "casino", "jackpot"],
        "scifi_mystery": ["space", "sci", "mystery", "horror", "alien", "planet", "moon", "galaxy"],
        "seasonal_misc": ["winter", "fire", "heat", "holiday", "christmas", "number", "lucky", "superstition", "car", "cars"],
        "format_non_theme": ["video", "stepper", "reel", "multigame", "multi game", "etg", "bank", "flex", "fusion", "sync", "renegade", "barcrest"],
    }
    for bucket, keywords in buckets.items():
        if any(re.search(rf"\b{re.escape(keyword)}\b", text) for keyword in keywords):
            return bucket
    return "uncategorized_theme"


def first_release_date(row):
    candidates = []
    for column in ["na_release_date", "lac_release_date", "emea_release_date", "apac_release_date", "matched_release_date"]:
        if column in row and pd.notna(row[column]):
            candidates.append(row[column])
    if not candidates:
        return pd.NaT
    return min(candidates)


attr_work = attr.copy()
attr_work["join_attr_theme_name"] = attr_work["theme_name_friendly"].map(norm_text)
attr_work = attr_work[attr_work["join_attr_theme_name"].notna()].copy()
if "audit_is_deleted" in attr_work.columns:
    attr_work = attr_work[~bool_series(attr_work["audit_is_deleted"]).fillna(0).astype(bool)].copy()

for column in ["na_release_date", "lac_release_date", "emea_release_date", "apac_release_date"]:
    if column in attr_work.columns:
        attr_work[column] = pd.to_datetime(attr_work[column], errors="coerce")

profiles = attr_work.merge(rel_profile, on="join_attr_theme_name", how="inner")
profiles = profiles.reset_index(drop=True)
profiles.insert(0, "profile_id", np.arange(len(profiles), dtype=int))

profiles["profile_release_date"] = profiles.apply(first_release_date, axis=1)
reference_date = profiles["profile_release_date"].max()
if pd.isna(reference_date):
    reference_date = pd.Timestamp.today().normalize()
profiles["release_age_months"] = profiles["profile_release_date"].map(lambda value: months_between(reference_date, value)).clip(lower=0)
profiles["release_age_bucket"] = profiles["release_age_months"].map(release_age_bucket)
profiles["attr_theme_bucket"] = profiles.apply(derive_attr_theme_bucket, axis=1)

for matrix_token in ["bingo", "hhr", "lottery", "prize", "reels"]:
    profiles[f"attr_matrix_token_{matrix_token}"] = profiles["game_matrix"].map(
        lambda value, token=matrix_token: float(any(token in item.lower() for item in parse_listish(value)))
    )

print(f"Attr-only profiles after accepted relationship join: {len(profiles):,}")
display(profiles.head(5))

In [0]:
holdout_n = max(MIN_HOLDOUT_ROWS, math.ceil(len(profiles) * HOLDOUT_FRACTION))
holdout_indices = set(
    profiles.assign(_release_sort=profiles["profile_release_date"])
    .sort_values("_release_sort", ascending=False, na_position="last")
    .head(holdout_n)
    .index
)

profiles["is_holdout"] = profiles.index.isin(holdout_indices)
profiles["is_qualified_historical"] = ~profiles["is_holdout"]
profiles["role"] = np.select(
    [profiles["is_holdout"], profiles["is_qualified_historical"]],
    ["holdout_new_game", "qualified_historical"],
    default="historical_not_qualified",
)

print(profiles["role"].value_counts().to_string())
print(
    "Holdout release range:",
    profiles.loc[profiles["is_holdout"], "profile_release_date"].min(),
    "to",
    profiles.loc[profiles["is_holdout"], "profile_release_date"].max(),
)

In [0]:
ATTR_CATEGORICAL_WEIGHTS = {
    "source_company_cd": 0.7,
    "source_system_cd": 0.35,
    "family_name": 1.7,
    "brand_name": 1.5,
    "math_model_family": 2.1,
    "product_family": 1,
    "game_type": 2.3,
    "game_matrix": 2.5,
    "product_segment": 1.8,
    "product_line": 2.3,
    "business_segment": 1.4,
    "gaming_channel_cd": 1.0,
    "volatility_cd": 1.2,
    "progressive_tiers": 1.1,
    "cert_family_ref": 0.9,
    "attr_theme_bucket": 1.7,
    "release_age_bucket": 0.30,
}

ATTR_BOOLEAN_WEIGHTS = {
    "is_wap": 2.2,
    "is_poker": 1.9,
    "is_mlp": 1.6,
    "is_multi_game": 2.6,
    "is_tournament_capable": 1.2,
    "is_multi_denom": 1.9,
    "game_matrix_bingo": 2.0,
    "game_matrix_hhr_exacta": 2.0,
    "game_matrix_lottery": 2.0,
    "game_matrix_prize_first_slot": 2.0,
    "game_matrix_reels_first_slot": 2.0,
    "attr_matrix_token_bingo": 1.4,
    "attr_matrix_token_hhr": 1.4,
    "attr_matrix_token_lottery": 1.4,
    "attr_matrix_token_prize": 1.4,
    "attr_matrix_token_reels": 1.4,
}

ATTR_NUMERIC_WEIGHTS = {
    "rtp_default": 0.9,
    "max_bet": 1.2,
    "min_bet": 0.9,
    "lines": 1.0,
    "ways": 1.0,
    "release_age_months": 0.30,
}

TEXT_SOURCE_COLUMNS = [
    "theme_name",
    "theme_name_friendly",
    "family_name",
    "brand_name",
    "product_family",
    "game_type",
    "product_line",
]


def add_categorical_block(frame, column, weight, max_levels=180, min_count=2):
    if column not in frame.columns:
        return None
    series = frame[column].map(clean_category).astype("string")
    counts = series.value_counts(dropna=False)
    if len(counts) <= 1:
        return None
    keep = set(counts[counts >= min_count].head(max_levels).index)
    bucketed = series.where(series.isin(keep), "__other__")
    dummies = pd.get_dummies(bucketed, prefix=f"cat__{column}", dtype=float)
    if dummies.shape[1] <= 1:
        return None
    return dummies * float(weight) * ATTR_FEATURE_MULTIPLIER


def add_boolean_block(frame, column, weight):
    if column not in frame.columns:
        return None
    values = bool_series(frame[column]).fillna(0.0)
    if values.nunique(dropna=False) <= 1:
        return None
    return pd.DataFrame({f"bool__{column}": values * float(weight) * ATTR_FEATURE_MULTIPLIER}, index=frame.index)


def add_numeric_block(frame, column, weight):
    if column not in frame.columns:
        return None
    values = numeric_series(frame[column])
    if any(token in column for token in ["bet", "lines", "ways", "months"]):
        values = np.log1p(values.clip(lower=0))
    scaled = robust_scale_01(values)
    if scaled is None or scaled.nunique(dropna=False) <= 1:
        return None
    return pd.DataFrame({f"num__{column}": scaled * float(weight) * ATTR_FEATURE_MULTIPLIER}, index=frame.index)


def add_text_token_block(frame, columns, max_tokens=140, min_count=2, max_doc_fraction=0.55):
    texts = []
    for _, row in frame.iterrows():
        text = " ".join(str(row.get(column, "")) for column in columns if pd.notna(row.get(column, pd.NA)))
        texts.append(tokenize_text(text))

    doc_freq = {}
    for tokens in texts:
        for token in set(tokens):
            doc_freq[token] = doc_freq.get(token, 0) + 1

    max_count = max(2, int(len(frame) * max_doc_fraction))
    selected = [
        token for token, count in sorted(doc_freq.items(), key=lambda item: (-item[1], item[0]))
        if count >= min_count and count <= max_count
    ][:max_tokens]

    if not selected:
        return None

    data = np.zeros((len(frame), len(selected)), dtype=float)
    for row_idx, tokens in enumerate(texts):
        token_set = set(tokens)
        for col_idx, token in enumerate(selected):
            if token in token_set:
                data[row_idx, col_idx] = 1.0
    return pd.DataFrame(data * TEXT_TOKEN_WEIGHT * ATTR_FEATURE_MULTIPLIER, columns=[f"token__{token}" for token in selected], index=frame.index)


def build_weighted_feature_matrix(frame):
    blocks = []
    feature_sources = {}

    for column, weight in ATTR_CATEGORICAL_WEIGHTS.items():
        block = add_categorical_block(frame, column, weight)
        if block is not None:
            blocks.append(block)
            feature_sources.update({feature: "attr" for feature in block.columns})

    for column, weight in ATTR_BOOLEAN_WEIGHTS.items():
        block = add_boolean_block(frame, column, weight)
        if block is not None:
            blocks.append(block)
            feature_sources.update({feature: "attr" for feature in block.columns})

    for column, weight in ATTR_NUMERIC_WEIGHTS.items():
        block = add_numeric_block(frame, column, weight)
        if block is not None:
            blocks.append(block)
            feature_sources.update({feature: "attr" for feature in block.columns})

    text_block = add_text_token_block(frame, TEXT_SOURCE_COLUMNS, max_tokens=MAX_TEXT_TOKENS)
    if text_block is not None:
        blocks.append(text_block)
        feature_sources.update({feature: "attr_text" for feature in text_block.columns})

    if not blocks:
        raise ValueError("No usable attr-only clustering features were created.")

    matrix = pd.concat(blocks, axis=1).fillna(0.0)
    matrix["__profile_bias"] = PROFILE_BIAS_WEIGHT
    feature_sources["__profile_bias"] = "bias"
    return matrix, feature_sources


feature_matrix, feature_sources = build_weighted_feature_matrix(profiles)
print(f"Feature matrix shape: {feature_matrix.shape[0]:,} profiles x {feature_matrix.shape[1]:,} attr-only weighted features")
print(pd.Series(feature_sources).value_counts().to_string())
print(f"Zero-norm rows: {(np.linalg.norm(feature_matrix.to_numpy(dtype=float), axis=1) == 0).sum()}")
display(feature_matrix.head(3))

In [0]:
X = feature_matrix.to_numpy(dtype=float)
condensed_distances = pdist(X, metric=DISTANCE_METRIC)
if not np.isfinite(condensed_distances).all():
    raise ValueError("Pairwise distances contain NaN or infinite values.")

distance_matrix = squareform(condensed_distances)
linkage_matrix = linkage(condensed_distances, method=LINKAGE_METHOD, optimal_ordering=True)

row_norms = np.linalg.norm(X, axis=1, keepdims=True)
X_unit = X / np.where(row_norms == 0, 1, row_norms)


def within_cluster_sse(x_values, labels):
    total = 0.0
    for label in np.unique(labels):
        cluster = x_values[labels == label]
        if len(cluster) == 0:
            continue
        center = cluster.mean(axis=0, keepdims=True)
        total += float(((cluster - center) ** 2).sum())
    return total


def elbow_line_distance(xs, ys):
    points = np.column_stack([xs, ys])
    start = points[0]
    end = points[-1]
    line = end - start
    denom = np.linalg.norm(line)
    if denom == 0:
        return np.zeros(len(points))
    return np.abs(np.cross(line, start - points)) / denom


rows = []
max_k = min(ELBOW_MAX_CLUSTERS, len(profiles) - 1)
for requested_k in range(ELBOW_MIN_CLUSTERS, max_k + 1):
    labels = fcluster(linkage_matrix, t=requested_k, criterion="maxclust")
    actual_k = len(np.unique(labels))
    sse = within_cluster_sse(X_unit, labels)
    counts = pd.Series(labels).value_counts()
    rows.append({
        "requested_k": requested_k,
        "actual_k": actual_k,
        "within_cluster_sse": sse,
        "min_cluster_size": int(counts.min()),
        "max_cluster_size": int(counts.max()),
    })

elbow_scores = pd.DataFrame(rows)
elbow_scores["sse_drop"] = elbow_scores["within_cluster_sse"].shift(1) - elbow_scores["within_cluster_sse"]
distances_to_line = elbow_line_distance(
    elbow_scores["actual_k"].to_numpy(dtype=float),
    elbow_scores["within_cluster_sse"].to_numpy(dtype=float),
)
elbow_scores["elbow_distance_to_line"] = distances_to_line
auto_elbow_k = int(elbow_scores.loc[elbow_scores["elbow_distance_to_line"].idxmax(), "actual_k"])

selected_k = ELBOW_SELECTED_N_CLUSTERS
if selected_k < ELBOW_MIN_CLUSTERS or selected_k > max_k:
    raise ValueError(f"ELBOW_SELECTED_N_CLUSTERS={selected_k} is outside the valid range.")

labels = fcluster(linkage_matrix, t=selected_k, criterion="maxclust")
counts = pd.Series(labels).value_counts().sort_values(ascending=False)
remap = {old_label: new_label for new_label, old_label in enumerate(counts.index, start=1)}
profiles["cluster_id"] = np.array([remap[label] for label in labels], dtype=int)

print(f"Auto elbow suggestion: {auto_elbow_k}")
print(f"Selected clusters used for this run: {profiles['cluster_id'].nunique()} (ELBOW_SELECTED_N_CLUSTERS={ELBOW_SELECTED_N_CLUSTERS})")
display(elbow_scores)
display(profiles["cluster_id"].value_counts().sort_index().rename("profiles_per_cluster"))

In [0]:
def find_holdout_like_games(profiles, linkage_matrix, distance_matrix):
    n = len(profiles)
    target_indices = profiles.index[profiles["is_holdout"]].tolist()
    historical_indices = set(profiles.index[profiles["is_qualified_historical"]].tolist())

    clusters = {idx: {idx} for idx in range(n)}
    merge_records = []
    for merge_step, row in enumerate(linkage_matrix, start=1):
        left = int(row[0])
        right = int(row[1])
        members = clusters[left] | clusters[right]
        node_id = n + merge_step - 1
        clusters[node_id] = members
        merge_records.append({
            "merge_step": merge_step,
            "node_id": node_id,
            "merge_distance": float(row[2]),
            "members": members,
        })

    rows = []
    for target_idx in target_indices:
        selected = None
        for record in merge_records:
            if target_idx not in record["members"]:
                continue
            qualifying = sorted(record["members"] & historical_indices)
            if qualifying:
                selected = {**record, "qualifying": qualifying}
                break

        if selected is None:
            continue

        ranked = sorted(selected["qualifying"], key=lambda candidate_idx: distance_matrix[target_idx, candidate_idx])
        if MAX_LIKE_GAMES_PER_TARGET is not None:
            ranked = ranked[:MAX_LIKE_GAMES_PER_TARGET]

        target = profiles.loc[target_idx]
        for rank, like_idx in enumerate(ranked, start=1):
            like = profiles.loc[like_idx]
            pair_distance = float(distance_matrix[target_idx, like_idx])
            similarity_score = float(np.clip(1.0 - pair_distance, 0.0, 1.0))
            merge_similarity = float(np.clip(1.0 - selected["merge_distance"], 0.0, 1.0))
            rows.append({
                "target_profile_id": int(target["profile_id"]),
                "target_theme_sk": target.get("theme_sk"),
                "target_theme_name_friendly": target.get("theme_name_friendly"),
                "target_product_family": target.get("product_family"),
                "target_product_line": target.get("product_line"),
                "target_game_type": target.get("game_type"),
                "target_game_matrix": target.get("game_matrix"),
                "target_release_date": target.get("profile_release_date"),
                "target_cluster_id": int(target.get("cluster_id")),
                "target_historical_game_name": target.get("best_historical_game_name"),
                "like_profile_id": int(like["profile_id"]),
                "like_theme_sk": like.get("theme_sk"),
                "like_theme_name_friendly": like.get("theme_name_friendly"),
                "like_product_family": like.get("product_family"),
                "like_product_line": like.get("product_line"),
                "like_game_type": like.get("game_type"),
                "like_game_matrix": like.get("game_matrix"),
                "like_release_date": like.get("profile_release_date"),
                "like_cluster_id": int(like.get("cluster_id")),
                "like_historical_game_name": like.get("best_historical_game_name"),
                "candidate_rank": rank,
                "pair_cosine_distance": pair_distance,
                "similarity_score": similarity_score,
                "merge_distance": selected["merge_distance"],
                "merge_similarity_score": merge_similarity,
                "smallest_qualifying_cluster_node_id": int(selected["node_id"]),
                "smallest_qualifying_cluster_size": len(selected["members"]),
                "qualifying_historical_in_cluster": len(selected["qualifying"]),
                "same_flat_cluster": bool(target.get("cluster_id") == like.get("cluster_id")),
            })

    return pd.DataFrame(rows)


like_games = find_holdout_like_games(profiles, linkage_matrix, distance_matrix)
if like_games.empty:
    raise ValueError("No like-game mappings were produced.")

grouped_like_games = (
    like_games.sort_values(["target_profile_id", "candidate_rank"])
    .groupby([
        "target_profile_id",
        "target_theme_sk",
        "target_theme_name_friendly",
        "target_product_family",
        "target_product_line",
        "target_game_type",
        "target_release_date",
        "target_cluster_id",
    ], dropna=False)
    .agg(
        like_game_count=("like_profile_id", "size"),
        best_similarity_score=("similarity_score", "max"),
        mean_similarity_score=("similarity_score", "mean"),
        like_attr_themes=("like_theme_name_friendly", lambda values: " | ".join(values.astype(str))),
        like_historical_games=("like_historical_game_name", lambda values: " | ".join(values.astype(str))),
        like_profile_ids=("like_profile_id", lambda values: " | ".join(values.astype(str))),
    )
    .reset_index()
    .sort_values(["best_similarity_score", "like_game_count"], ascending=[False, False])
)

print(f"Mapping rows: {len(like_games):,}")
print(f"Mapped holdout targets: {like_games['target_profile_id'].nunique():,} / {profiles['is_holdout'].sum():,}")
display(like_games.sort_values(["target_profile_id", "candidate_rank"]).head(30))
display(grouped_like_games.head(20))

In [0]:
def top_values(series, n=3):
    counts = series.map(clean_category).value_counts(normalize=True).head(n)
    if counts.empty:
        return ""
    return "; ".join(f"{idx} ({share:.0%})" for idx, share in counts.items())


def build_cluster_summary(profiles, feature_matrix):
    feature_view = feature_matrix.drop(columns=["__profile_bias"], errors="ignore")
    cluster_means = feature_view.groupby(profiles["cluster_id"]).mean()
    global_means = feature_view.mean()
    feature_lift = cluster_means.subtract(global_means, axis=1)

    summary = (
        profiles.groupby("cluster_id", as_index=False)
        .agg(
            profile_count=("profile_id", "size"),
            holdout_count=("is_holdout", "sum"),
            historical_count=("is_qualified_historical", "sum"),
            min_release_date=("profile_release_date", "min"),
            max_release_date=("profile_release_date", "max"),
            median_release_age_months=("release_age_months", "median"),
        )
        .sort_values("cluster_id")
    )

    characteristic_columns = [
        "product_family",
        "product_line",
        "game_type",
        "game_matrix",
        "math_model_family",
        "product_segment",
        "family_name",
        "brand_name",
        "business_segment",
        "volatility_cd",
        "attr_theme_bucket",
        "is_multi_denom",
        "is_wap",
        "is_poker",
    ]
    for column in characteristic_columns:
        if column in profiles.columns:
            values = profiles.groupby("cluster_id")[column].apply(top_values).rename(f"top_{column}")
            summary = summary.merge(values.reset_index(), on="cluster_id", how="left")

    top_features = []
    for cluster_id in summary["cluster_id"]:
        lifted = feature_lift.loc[cluster_id].sort_values(ascending=False).head(TOP_CLUSTER_FEATURES)
        top_features.append("; ".join(pretty_feature_name(name) for name in lifted.index[:8]))
    summary["top_distinguishing_features"] = top_features
    return summary, feature_lift


cluster_summary, feature_lift = build_cluster_summary(profiles, feature_matrix)
display(cluster_summary)

In [0]:
def plot_elbow(elbow_scores):
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(elbow_scores["actual_k"], elbow_scores["within_cluster_sse"], marker="o", linewidth=2)
    auto_k = int(elbow_scores.loc[elbow_scores["elbow_distance_to_line"].idxmax(), "actual_k"])
    selected_k = profiles["cluster_id"].nunique()
    ax.axvline(auto_k, color="#777777", linestyle="--", linewidth=1.2, label=f"Auto elbow: {auto_k}")
    ax.axvline(selected_k, color="#111111", linestyle="-", linewidth=1.4, label=f"Selected: {selected_k}")
    ax.set_title("Elbow Method On Attr-Only Features")
    ax.set_xlabel("Number of clusters")
    ax.set_ylabel("Within-cluster SSE on normalized attr matrix")
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)
    path = OUTPUT_DIR / "elbow_method.png"
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.show()
    return path


def classical_mds(distance_matrix, n_components=2):
    n = distance_matrix.shape[0]
    squared = distance_matrix ** 2
    centering = np.eye(n) - np.ones((n, n)) / n
    gram = -0.5 * centering @ squared @ centering
    eigvals, eigvecs = np.linalg.eigh(gram)
    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]
    positive = np.maximum(eigvals[:n_components], 0)
    coords = eigvecs[:, :n_components] * np.sqrt(positive)
    if coords.shape[1] < n_components:
        coords = np.pad(coords, ((0, 0), (0, n_components - coords.shape[1])))
    return coords


def plot_dendrogram_view():
    selected_k = profiles["cluster_id"].nunique()
    cut_distance = linkage_matrix[-(selected_k - 1), 2] if selected_k > 1 else linkage_matrix[-1, 2]
    labels = [
        f"{int(row.profile_id)} | {str(row.theme_name_friendly)[:34]} | {str(row.product_family)[:20]}"
        for row in profiles.itertuples(index=False)
    ]
    fig, ax = plt.subplots(figsize=(18, 9))
    kwargs = {
        "Z": linkage_matrix,
        "labels": labels,
        "leaf_rotation": 90,
        "leaf_font_size": 7,
        "color_threshold": cut_distance,
        "show_contracted": True,
        "ax": ax,
    }
    if len(profiles) > MAX_DENDROGRAM_LEAVES:
        kwargs.update({"truncate_mode": "lastp", "p": MAX_DENDROGRAM_LEAVES})
    dendrogram(**kwargs)
    ax.axhline(cut_distance, color="black", linestyle="--", linewidth=1, alpha=0.7)
    ax.set_title("Attr-Only Complete-Linkage Dendrogram")
    ax.set_xlabel("Attr theme profile")
    ax.set_ylabel(f"{DISTANCE_METRIC.title()} distance")
    path = OUTPUT_DIR / "dendrogram_selected_clusters.png"
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.show()
    return path


def plot_projection():
    coords = classical_mds(distance_matrix)
    profiles["mds_x"] = coords[:, 0]
    profiles["mds_y"] = coords[:, 1]
    fig, ax = plt.subplots(figsize=(10, 7))
    cmap = plt.get_cmap("tab10")
    for idx, cluster_id in enumerate(sorted(profiles["cluster_id"].unique())):
        mask = profiles["cluster_id"].eq(cluster_id)
        historical = mask & ~profiles["is_holdout"]
        holdout = mask & profiles["is_holdout"]
        color = cmap(idx % cmap.N)
        ax.scatter(profiles.loc[historical, "mds_x"], profiles.loc[historical, "mds_y"], s=24, alpha=0.50, color=color, label=f"Cluster {cluster_id}")
        ax.scatter(profiles.loc[holdout, "mds_x"], profiles.loc[holdout, "mds_y"], s=72, marker="x", linewidths=1.7, color=color)
    ax.set_title("Attr-Only 2D Projection (x = Holdout)")
    ax.set_xlabel("MDS 1")
    ax.set_ylabel("MDS 2")
    ax.grid(alpha=0.20)
    ax.legend(ncol=2, fontsize=8, frameon=False)
    path = OUTPUT_DIR / "cluster_projection_mds.png"
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.show()
    return path


elbow_plot_path = plot_elbow(elbow_scores)
dendrogram_path = plot_dendrogram_view()
projection_path = plot_projection()
print("Saved overview visuals:")
for path in [elbow_plot_path, dendrogram_path, projection_path]:
    print(f"- {path}")

In [0]:
def save_cluster_feature_chart(cluster_id):
    lifted = feature_lift.loc[cluster_id].sort_values(ascending=False).head(TOP_CLUSTER_FEATURES)
    labels = [pretty_feature_name(name) for name in lifted.index]
    values = lifted.to_numpy()
    cluster_rows = profiles[profiles["cluster_id"].eq(cluster_id)]
    summary_row = cluster_summary[cluster_summary["cluster_id"].eq(cluster_id)].iloc[0]

    fig, ax = plt.subplots(figsize=(11, 7))
    y = np.arange(len(values))
    ax.barh(y, values, color="#3568a8")
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel("Feature lift vs global mean")
    ax.set_title(f"Cluster {cluster_id}: Attr Features ({len(cluster_rows)} games, {int(summary_row['holdout_count'])} holdout)")
    ax.grid(axis="x", alpha=0.20)

    text_lines = [
        f"Top product family: {summary_row.get('top_product_family', '')}",
        f"Top product line: {summary_row.get('top_product_line', '')}",
        f"Top game type: {summary_row.get('top_game_type', '')}",
        f"Top matrix: {summary_row.get('top_game_matrix', '')}",
    ]
    fig.text(0.02, 0.02, "\\n".join(text_lines), ha="left", va="bottom", fontsize=9)
    path = CLUSTER_CHART_DIR / f"cluster_{int(cluster_id):02d}_features.png"
    fig.tight_layout(rect=(0, 0.10, 1, 1))
    fig.savefig(path, dpi=160)
    plt.close(fig)
    return path


cluster_chart_paths = []
for cluster_id in sorted(profiles["cluster_id"].unique()):
    cluster_chart_paths.append(save_cluster_feature_chart(cluster_id))

print(f"Saved {len(cluster_chart_paths)} cluster feature charts to {CLUSTER_CHART_DIR}")
display(pd.DataFrame({"cluster_id": sorted(profiles["cluster_id"].unique()), "chart_path": [str(p) for p in cluster_chart_paths]}))

In [0]:
def top_active_features(profile_index, n=TOP_HOLDOUT_FEATURES):
    values = feature_matrix.iloc[profile_index].drop(labels=["__profile_bias"], errors="ignore")
    active = values[values > 0].sort_values(ascending=False).head(n)
    return active


def save_holdout_feature_chart(profile_index):
    row = profiles.loc[profile_index]
    active = top_active_features(profile_index)
    target_rows = like_games[like_games["target_profile_id"].eq(row["profile_id"])].sort_values("candidate_rank").head(5)
    like_text = "; ".join(
        f"{item.like_theme_name_friendly} ({item.similarity_score:.3f})"
        for item in target_rows.itertuples(index=False)
    )
    if not like_text:
        like_text = "No like games found"

    fig, (ax_text, ax_bar) = plt.subplots(
        2,
        1,
        figsize=(12, 8),
        gridspec_kw={"height_ratios": [1.1, 2.2]},
    )
    ax_text.axis("off")
    text = (
        f"Holdout: {row.get('theme_name_friendly')}\\n"
        f"Assigned cluster: {int(row.get('cluster_id'))}\\n"
        f"Product family: {row.get('product_family')} | Product line: {row.get('product_line')}\\n"
        f"Game type: {row.get('game_type')} | Game matrix: {row.get('game_matrix')}\\n"
        f"Release date: {row.get('profile_release_date')} | Historical mapping: {row.get('best_historical_game_name')}\\n"
        f"Top like games: {like_text}"
    )
    ax_text.text(0.01, 0.98, text, va="top", ha="left", fontsize=10, wrap=True)

    labels = [pretty_feature_name(name) for name in active.index]
    values = active.to_numpy()
    y = np.arange(len(values))
    ax_bar.barh(y, values, color="#2f7f63")
    ax_bar.set_yticks(y)
    ax_bar.set_yticklabels(labels, fontsize=9)
    ax_bar.invert_yaxis()
    ax_bar.set_xlabel("Weighted active attr feature value")
    ax_bar.set_title("Strongest Active Attr Features For This Holdout")
    ax_bar.grid(axis="x", alpha=0.20)

    file_name = f"holdout_{int(row['profile_id']):04d}_cluster_{int(row['cluster_id']):02d}_{safe_filename(row.get('theme_name_friendly'))}.png"
    path = HOLDOUT_CHART_DIR / file_name
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)
    return path


holdout_chart_rows = []
for profile_index in profiles.index[profiles["is_holdout"]]:
    path = save_holdout_feature_chart(profile_index)
    row = profiles.loc[profile_index]
    holdout_chart_rows.append({
        "profile_id": int(row["profile_id"]),
        "theme_name_friendly": row.get("theme_name_friendly"),
        "cluster_id": int(row.get("cluster_id")),
        "chart_path": str(path),
    })

holdout_chart_index = pd.DataFrame(holdout_chart_rows).sort_values(["cluster_id", "theme_name_friendly"])
print(f"Saved {len(holdout_chart_index)} holdout feature charts to {HOLDOUT_CHART_DIR}")
display(holdout_chart_index.head(20))

In [0]:
profiles.to_csv(PROFILES_OUTPUT_PATH, index=False)
feature_matrix.to_csv(FEATURE_MATRIX_OUTPUT_PATH, index=False)
like_games.to_csv(MAPPING_OUTPUT_PATH, index=False)
grouped_like_games.to_csv(GROUPED_MAPPING_OUTPUT_PATH, index=False)
cluster_summary.to_csv(CLUSTER_SUMMARY_OUTPUT_PATH, index=False)
elbow_scores.to_csv(ELBOW_OUTPUT_PATH, index=False)
holdout_chart_index.to_csv(CHART_INDEX_OUTPUT_PATH, index=False)

manifest = {
    "inputs": {
        "attr_path": str(ATTR_PATH),
        "attr_raw_path": str(ATTR_RAW_PATH),
        "rel_map_path": str(REL_MAP_PATH),
    },
    "profile_rows": int(len(profiles)),
    "feature_columns": int(feature_matrix.shape[1]),
    "feature_source_counts": {str(k): int(v) for k, v in pd.Series(feature_sources).value_counts().items()},
    "holdout_profiles": int(profiles["is_holdout"].sum()),
    "historical_profiles": int(profiles["is_qualified_historical"].sum()),
    "mapped_holdout_profiles": int(like_games["target_profile_id"].nunique()),
    "mapping_rows": int(len(like_games)),
    "auto_elbow_k": int(auto_elbow_k),
    "selected_clusters": int(profiles["cluster_id"].nunique()),
    "selected_clusters_setting": int(ELBOW_SELECTED_N_CLUSTERS),
    "excluded_match_methods": sorted(EXCLUDED_MATCH_METHODS),
    "distance_metric": DISTANCE_METRIC,
    "linkage_method": LINKAGE_METHOD,
    "excluded_feature_families": ["cabinet_name", "performance"],
    "outputs": {
        "profiles": str(PROFILES_OUTPUT_PATH),
        "feature_matrix": str(FEATURE_MATRIX_OUTPUT_PATH),
        "mapping": str(MAPPING_OUTPUT_PATH),
        "grouped_mapping": str(GROUPED_MAPPING_OUTPUT_PATH),
        "cluster_summary": str(CLUSTER_SUMMARY_OUTPUT_PATH),
        "elbow_scores": str(ELBOW_OUTPUT_PATH),
        "chart_index": str(CHART_INDEX_OUTPUT_PATH),
        "cluster_chart_dir": str(CLUSTER_CHART_DIR),
        "holdout_chart_dir": str(HOLDOUT_CHART_DIR),
        "report_dir": str(OUTPUT_DIR),
    },
}
MANIFEST_OUTPUT_PATH.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

print("Saved outputs:")
for path in [
    PROFILES_OUTPUT_PATH,
    FEATURE_MATRIX_OUTPUT_PATH,
    MAPPING_OUTPUT_PATH,
    GROUPED_MAPPING_OUTPUT_PATH,
    CLUSTER_SUMMARY_OUTPUT_PATH,
    ELBOW_OUTPUT_PATH,
    CHART_INDEX_OUTPUT_PATH,
    MANIFEST_OUTPUT_PATH,
]:
    print(f"- {path}")